In [5]:
import duckdb
import pandas as pd
import numpy as np
from IPython.display import display

# ============================================================
# AML TRANSACTION MONITORING: HIGH VELOCITY / HIGH VOLUME SPIKE
# Dataset: paysim dataset.csv
# ============================================================

# SQL Sorgusu: Tüm verisetindeki yüksek hacimli ve çoklu işlemleri hedefler
sql_velocity_query = """
WITH customer_activity AS (
    SELECT
        nameOrig AS customer_id,
        COUNT(*) AS tx_count,
        SUM(amount) AS total_amount,
        AVG(amount) AS avg_amount,
        MAX(amount) AS max_amount
    FROM read_csv_auto('paysim dataset.csv')
    GROUP BY nameOrig
),

global_stats AS (
    SELECT 
        AVG(amount) AS global_avg_amount
    FROM read_csv_auto('paysim dataset.csv')
)

SELECT
    c.customer_id,
    c.tx_count AS tx_count_last_48h,
    ROUND(c.total_amount, 2) AS total_amount_last_48h,
    ROUND(c.avg_amount, 2) AS avg_transaction_amount_last_48h,
    ROUND(g.global_avg_amount * 24, 2) AS baseline_daily_volume,
    ROUND(c.total_amount / NULLIF(g.global_avg_amount * 24, 0), 2) AS spike_multiplier
FROM customer_activity c
CROSS JOIN global_stats g
WHERE 
    -- Eşikleri çok daha ulaşılır yaptık:
    c.total_amount >= 50000 
ORDER BY c.total_amount DESC
LIMIT 50
"""

# ------------------------------------------------------------
# SQL SORGUSUNU ÇALIŞTIR
# ------------------------------------------------------------
velocity_df = duckdb.query(sql_velocity_query).to_df()

print(f"✅ DuckDB ile tespit edilen toplam velocity alarmı: {len(velocity_df)}")

# ------------------------------------------------------------
# PANDAS İLE RISK SEGMENTASYONU
# ------------------------------------------------------------

velocity_df['risk_tier'] = pd.cut(
    velocity_df['spike_multiplier'],
    bins=[0, 10, np.inf],
    labels=['MEDIUM_VELOCITY_RISK', 'HIGH_VELOCITY_RISK'],
    right=False
)

velocity_df['manual_review_flag'] = np.where(
    velocity_df['spike_multiplier'] >= 10,
    'HIGH_PRIORITY_REVIEW',
    'STANDARD_REVIEW'
)

risk_summary = (
    velocity_df
    .groupby('risk_tier', observed=True)
    .agg(
        alert_count=('customer_id', 'count'),
        avg_amount_last_48h=('total_amount_last_48h', 'mean'),
        max_multiplier=('spike_multiplier', 'max'),
        avg_transaction_count=('tx_count_last_48h', 'mean')
    )
    .reset_index()
)

risk_summary['avg_amount_last_48h'] = risk_summary['avg_amount_last_48h'].round(2)
risk_summary['max_multiplier'] = risk_summary['max_multiplier'].round(2)
risk_summary['avg_transaction_count'] = risk_summary['avg_transaction_count'].round(2)

# ------------------------------------------------------------
# JUPYTER TABLO ÇIKTILARI
# ------------------------------------------------------------

print("\n--- 1. Risk Seviyesi Dağılımı ---")
display(risk_summary)

print("\n--- 2. En Yüksek Sıçrama Yapan Müşteriler (Top Cases) ---")
display(velocity_df.head(10))

print("\n--- 3. Manuel İnceleme Adayları (AML Review Candidates) ---")
display(
    velocity_df[[
        'customer_id', 
        'tx_count_last_48h', 
        'total_amount_last_48h', 
        'baseline_daily_volume', 
        'spike_multiplier', 
        'risk_tier', 
        'manual_review_flag'
    ]].head(10)
)

✅ DuckDB ile tespit edilen toplam velocity alarmı: 50

--- 1. Risk Seviyesi Dağılımı ---


,risk_tier,alert_count,avg_amount_last_48h,max_multiplier,avg_transaction_count
0,HIGH_VELOCITY_RISK,50,57813841.21,21.42,1.0



--- 2. En Yüksek Sıçrama Yapan Müşteriler (Top Cases) ---


,customer_id,tx_count_last_48h,total_amount_last_48h,avg_transaction_amount_last_48h,baseline_daily_volume,spike_multiplier,risk_tier,manual_review_flag
0,C1715283297,1,92445516.64,92445516.64,4316685.69,21.42,HIGH_VELOCITY_RISK,HIGH_PRIORITY_REVIEW
1,C2127282686,1,73823490.36,73823490.36,4316685.69,17.10,HIGH_VELOCITY_RISK,HIGH_PRIORITY_REVIEW
2,C2044643633,1,71172480.42,71172480.42,4316685.69,16.49,HIGH_VELOCITY_RISK,HIGH_PRIORITY_REVIEW
3,C1425667947,1,69886731.30,69886731.30,4316685.69,16.19,HIGH_VELOCITY_RISK,HIGH_PRIORITY_REVIEW
4,C1584456031,1,69337316.27,69337316.27,4316685.69,16.06,HIGH_VELOCITY_RISK,HIGH_PRIORITY_REVIEW
5,C811810230,1,67500761.29,67500761.29,4316685.69,15.64,HIGH_VELOCITY_RISK,HIGH_PRIORITY_REVIEW
6,C420748282,1,66761272.21,66761272.21,4316685.69,15.47,HIGH_VELOCITY_RISK,HIGH_PRIORITY_REVIEW
7,C1139847449,1,64234448.19,64234448.19,4316685.69,14.88,HIGH_VELOCITY_RISK,HIGH_PRIORITY_REVIEW
8,C300140823,1,63847992.58,63847992.58,4316685.69,14.79,HIGH_VELOCITY_RISK,HIGH_PRIORITY_REVIEW
9,C372535854,1,63294839.63,63294839.63,4316685.69,14.66,HIGH_VELOCITY_RISK,HIGH_PRIORITY_REVIEW



--- 3. Manuel İnceleme Adayları (AML Review Candidates) ---


,customer_id,tx_count_last_48h,total_amount_last_48h,baseline_daily_volume,spike_multiplier,risk_tier,manual_review_flag
0,C1715283297,1,92445516.64,4316685.69,21.42,HIGH_VELOCITY_RISK,HIGH_PRIORITY_REVIEW
1,C2127282686,1,73823490.36,4316685.69,17.10,HIGH_VELOCITY_RISK,HIGH_PRIORITY_REVIEW
2,C2044643633,1,71172480.42,4316685.69,16.49,HIGH_VELOCITY_RISK,HIGH_PRIORITY_REVIEW
3,C1425667947,1,69886731.30,4316685.69,16.19,HIGH_VELOCITY_RISK,HIGH_PRIORITY_REVIEW
4,C1584456031,1,69337316.27,4316685.69,16.06,HIGH_VELOCITY_RISK,HIGH_PRIORITY_REVIEW
5,C811810230,1,67500761.29,4316685.69,15.64,HIGH_VELOCITY_RISK,HIGH_PRIORITY_REVIEW
6,C420748282,1,66761272.21,4316685.69,15.47,HIGH_VELOCITY_RISK,HIGH_PRIORITY_REVIEW
7,C1139847449,1,64234448.19,4316685.69,14.88,HIGH_VELOCITY_RISK,HIGH_PRIORITY_REVIEW
8,C300140823,1,63847992.58,4316685.69,14.79,HIGH_VELOCITY_RISK,HIGH_PRIORITY_REVIEW
9,C372535854,1,63294839.63,4316685.69,14.66,HIGH_VELOCITY_RISK,HIGH_PRIORITY_REVIEW
